In [1]:
!pip install timm
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import numpy as np
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight

In [2]:
DATASET_PATH = "/kaggle/input/datasets/sekhsujonhaque/cape-gooseberry-classification-dataset/classification_dataset"

In [3]:
train_transforms = transforms.Compose([
    transforms.Resize((300,300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2,0.2,0.2,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((300,300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [4]:
train_dataset = datasets.ImageFolder(DATASET_PATH + "/train", transform=train_transforms)
val_dataset   = datasets.ImageFolder(DATASET_PATH + "/valid", transform=val_transforms)
test_dataset  = datasets.ImageFolder(DATASET_PATH + "/test", transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

print("Classes:", train_dataset.classes)

Classes: ['INTERMEDIA', 'MADURO', 'NO_APTA', 'VERDE']


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [6]:
labels = [label for _, label in train_dataset]

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [7]:
model = timm.create_model('efficientnet_b3', pretrained=True)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last few layers
for param in model.blocks[-5:].parameters():
    param.requires_grad = True

# Custom classifier
model.classifier = nn.Sequential(
    nn.BatchNorm1d(model.classifier.in_features),
    nn.Dropout(0.5),
    nn.Linear(model.classifier.in_features, 4)
)

model = model.to(device)

In [8]:
optimizer = optim.AdamW(model.parameters(), lr=0.00015)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [9]:
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [10]:
EPOCHS = 35
best_val = 0
patience = 5
counter = 0
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    val_acc = evaluate(val_loader)
    if val_acc > best_val:
        best_val = val_acc
        counter = 0
        torch.save(model.state_dict(), "/kaggle/working/best_model.pth")
        print("Best model saved")
    else:
        counter += 1
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}, Val Acc: {val_acc:.2f}%")
    if counter >= patience:
        print("Early stopping triggered")
        break

Best model saved
Epoch 1/35, Loss: 58.8396, Val Acc: 55.56%
Best model saved
Epoch 2/35, Loss: 42.3918, Val Acc: 67.25%
Best model saved
Epoch 3/35, Loss: 31.5130, Val Acc: 72.51%
Epoch 4/35, Loss: 25.9289, Val Acc: 71.93%
Epoch 5/35, Loss: 19.5126, Val Acc: 70.76%
Epoch 6/35, Loss: 15.0130, Val Acc: 72.51%
Epoch 7/35, Loss: 13.3363, Val Acc: 72.51%
Best model saved
Epoch 8/35, Loss: 12.0451, Val Acc: 73.68%
Epoch 9/35, Loss: 9.2202, Val Acc: 73.10%
Best model saved
Epoch 10/35, Loss: 9.0691, Val Acc: 74.27%
Best model saved
Epoch 11/35, Loss: 6.0753, Val Acc: 74.85%
Epoch 12/35, Loss: 5.8586, Val Acc: 74.27%
Epoch 13/35, Loss: 6.1019, Val Acc: 74.27%
Best model saved
Epoch 14/35, Loss: 5.1038, Val Acc: 75.44%
Epoch 15/35, Loss: 5.0243, Val Acc: 74.85%
Epoch 16/35, Loss: 4.4021, Val Acc: 73.68%
Epoch 17/35, Loss: 4.0425, Val Acc: 74.27%
Epoch 18/35, Loss: 3.6896, Val Acc: 73.10%
Epoch 19/35, Loss: 3.9607, Val Acc: 74.85%
Early stopping triggered


In [11]:
model.load_state_dict(torch.load("/kaggle/working/best_model.pth"))
model.eval()
test_acc = evaluate(test_loader)
print("FINAL TEST ACCURACY:", test_acc)

FINAL TEST ACCURACY: 75.0
